# 1. Google Colab Setup & Drive Mount
Mounting Google Drive to ensure persistent storage of models and checkpoints.

In [ ]:
import os
import sys

# Google Drive Mount (Only executes if in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_ROOT = '/content/drive/MyDrive/KokBisaResearch'
    os.makedirs(COLAB_ROOT, exist_ok=True)
    os.chdir(COLAB_ROOT)
    print(f"Mounted Google Drive. Working directory set to: {os.getcwd()}")
except ImportError:
    print("Not running in Google Colab. Using local environment.")
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    print(f"Working directory set to: {os.getcwd()}")


# 2. Environment Setup & Imports

In [ ]:
# ==============================================================
# FULL 5-EXPERIMENT BENCHMARK
# IndoBERT Optimization
# 8-Class Discourse Classification
# Google Colab / Tesla T4
# ==============================================================

import os
import gc
import json
import time
import shutil
import inspect

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)


# 3. Label Configuration

In [ ]:
LABEL2ID = {
    "Question": 0,
    "Opinion": 1,
    "Disagreement": 2,
    "Correction": 3,
    "Suggestion": 4,
    "Praise": 5,
    "Agreement": 6,
    "Experience": 7
}

ID2LABEL = {
    0: "Question",
    1: "Opinion",
    2: "Disagreement",
    3: "Correction",
    4: "Suggestion",
    5: "Praise",
    6: "Agreement",
    7: "Experience"
}

NUM_LABELS = len(LABEL2ID)

print("=" * 70)
print("LABEL CONFIGURATION")
print("=" * 70)

print(f"Number of labels: {NUM_LABELS}\n")

for label_id, label_name in ID2LABEL.items():
    print(f"{label_id}: {label_name}")


# 4. Dataset Loading & Validation

In [ ]:
DATA_DIR = "data/processed/"
TRAIN_PATH = os.path.join(DATA_DIR, "train_balanced.parquet")
VAL_PATH = os.path.join(DATA_DIR, "validation_balanced.parquet")
TEST_PATH = os.path.join(DATA_DIR, "test_balanced.parquet")

df_train = pd.read_parquet(TRAIN_PATH)
df_val = pd.read_parquet(VAL_PATH)
df_test = pd.read_parquet(TEST_PATH)

print("\n" + "=" * 70)
print("DATASET VALIDATION")
print("=" * 70)

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)

def prepare_label_column(df):
    df = df.copy()
    if "discourse_label" in df.columns:
        if df["discourse_label"].dtype == object:
            df["label"] = df["discourse_label"].map(LABEL2ID)
        else:
            df["label"] = df["discourse_label"].astype(int)
    elif "label" in df.columns:
        if not pd.api.types.is_numeric_dtype(df["label"]):
            df["label"] = df["label"].map(LABEL2ID)
        else:
            df["label"] = df["label"].astype(int)
    else:
        raise ValueError("Dataset tidak memiliki kolom 'discourse_label' atau 'label'.")
    return df

df_train = prepare_label_column(df_train)
df_val = prepare_label_column(df_val)
df_test = prepare_label_column(df_test)

assert df_train["label"].isna().sum() == 0, "ERROR: Ada label kosong pada training dataset!"
assert df_val["label"].isna().sum() == 0, "ERROR: Ada label kosong pada validation dataset!"
assert df_train["label"].between(0, NUM_LABELS - 1).all(), "ERROR: Train memiliki label di luar range 0-7!"
assert df_val["label"].between(0, NUM_LABELS - 1).all(), "ERROR: Validation memiliki label di luar range 0-7!"

print("\nTrain Label Distribution")
for i in range(NUM_LABELS):
    count = (df_train["label"] == i).sum()
    print(f"{i} - {ID2LABEL[i]}: {count}")

print("\nValidation Label Distribution")
for i in range(NUM_LABELS):
    count = (df_val["label"] == i).sum()
    print(f"{i} - {ID2LABEL[i]}: {count}")


# 5. GPU Configuration

In [ ]:
print("\n" + "=" * 70)
print("GPU CONFIGURATION")
print("=" * 70)

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    print(f"GPU detected: {GPU_NAME}")
    USE_FP16 = True
    USE_BF16 = False
    print("Mixed precision: FP16")
else:
    print("WARNING: GPU NOT DETECTED")
    print("Running FP32")
    USE_FP16 = False
    USE_BF16 = False


# 6. Experiment Plan - 5 Experiments

In [ ]:
experiments = [
    {
        "experiment_id": "EXP_01",
        "model_name": "indobenchmark/indobert-base-p1",
        "learning_rate": 1e-5,
        "batch_size": 16,
        "gradient_accumulation_steps": 1,
        "epochs": 5,
        "max_length": 256
    },
    {
        "experiment_id": "EXP_02",
        "model_name": "indobenchmark/indobert-base-p1",
        "learning_rate": 2e-5,
        "batch_size": 16,
        "gradient_accumulation_steps": 1,
        "epochs": 5,
        "max_length": 256
    },
    {
        "experiment_id": "EXP_03",
        "model_name": "indobenchmark/indobert-base-p1",
        "learning_rate": 3e-5,
        "batch_size": 16,
        "gradient_accumulation_steps": 1,
        "epochs": 5,
        "max_length": 256
    },
    {
        "experiment_id": "EXP_04",
        "model_name": "indobenchmark/indobert-base-p1",
        "learning_rate": 2e-5,
        "batch_size": 32,
        "gradient_accumulation_steps": 1,
        "epochs": 5,
        "max_length": 256
    },
    {
        "experiment_id": "EXP_05",
        "model_name": "indobenchmark/indobert-base-p1",
        "learning_rate": 2e-5,
        "batch_size": 16,
        "gradient_accumulation_steps": 1,
        "epochs": 7,
        "max_length": 256
    }
]

print("\n" + "=" * 70)
print("EXPERIMENT PLAN")
print("=" * 70)

for exp in experiments:
    effective_batch = exp["batch_size"] * exp["gradient_accumulation_steps"]
    print(f"\n{exp['experiment_id']}")
    print(f"Model       : {exp['model_name']}")
    print(f"LR          : {exp['learning_rate']}")
    print(f"Batch Size  : {exp['batch_size']}")
    print(f"Grad Accum  : {exp['gradient_accumulation_steps']}")
    print(f"Effective Batch: {effective_batch}")
    print(f"Epochs      : {exp['epochs']}")
    print(f"Max Length  : {exp['max_length']}")


# 7. Output Directories & Data Preparation

In [ ]:
MODEL_DIR = "outputs/training/temp_models"
FINAL_DIR = "outputs/training"
BEST_MODEL_EXPORT_DIR = "outputs/training/best_model"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_EXPORT_DIR, exist_ok=True)

possible_text_columns = ["comment_text", "text", "comment", "cleaned_text", "content"]
TEXT_COLUMN = None
for column in possible_text_columns:
    if column in df_train.columns:
        TEXT_COLUMN = column
        break

if TEXT_COLUMN is None:
    raise ValueError(f"Tidak ditemukan kolom teks.\nKolom tersedia: {list(df_train.columns)}")

print("\nText column detected:", TEXT_COLUMN)

def prepare_dataset(dataframe, tokenizer, max_length):
    dataset = Dataset.from_pandas(dataframe[[TEXT_COLUMN, "label"]].reset_index(drop=True))
    def tokenize_function(batch):
        return tokenizer(batch[TEXT_COLUMN], truncation=True, max_length=max_length)
    dataset = dataset.map(tokenize_function, batched=True)
    dataset = dataset.remove_columns([TEXT_COLUMN])
    dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    return dataset

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, predictions, average="macro", zero_division=0)
    weighted_f1 = f1_score(labels, predictions, average="weighted", zero_division=0)
    accuracy = accuracy_score(labels, predictions)
    macro_precision = precision_score(labels, predictions, average="macro", zero_division=0)
    macro_recall = recall_score(labels, predictions, average="macro", zero_division=0)
    return {
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall
    }


# 8. Execute Benchmark Loop

In [ ]:
print("\n" + "=" * 70)
print("STARTING 5-EXPERIMENT BENCHMARK")
print("=" * 70)

results = []
best_score = -1
best_result = None
best_experiment = None
best_model_path = None

for exp in experiments:
    print("\n" + "=" * 70)
    print(f"Running {exp['experiment_id']} ({exp['model_name']})")
    print("=" * 70)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    experiment_dir = os.path.join(MODEL_DIR, exp["experiment_id"])
    os.makedirs(experiment_dir, exist_ok=True)

    print("\nLoading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(exp["model_name"], use_fast=True)

    print("\nPreparing datasets...")
    ds_train = prepare_dataset(df_train, tokenizer, exp["max_length"])
    ds_val = prepare_dataset(df_val, tokenizer, exp["max_length"])

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    print("\nLoading model...")
    model = AutoModelForSequenceClassification.from_pretrained(
        exp["model_name"],
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True
    )

    model.config.num_labels = NUM_LABELS
    model.config.id2label = ID2LABEL
    model.config.label2id = LABEL2ID

    training_args = TrainingArguments(
        output_dir=experiment_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=exp["learning_rate"],
        per_device_train_batch_size=exp["batch_size"],
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=exp["gradient_accumulation_steps"],
        num_train_epochs=exp["epochs"],
        weight_decay=0.01,
        logging_strategy="steps",
        logging_steps=25,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=USE_FP16,
        bf16=USE_BF16,
        report_to="none",
        seed=42,
        data_seed=42
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    print("\nStarting training...")
    start_time = time.time()
    trainer.train()
    training_time = time.time() - start_time

    print("\nEvaluating validation set...")
    eval_result = trainer.evaluate()
    best_checkpoint = trainer.state.best_model_checkpoint

    experiment_result = {
        "experiment_id": exp["experiment_id"],
        "model": exp["model_name"],
        "learning_rate": exp["learning_rate"],
        "batch_size": exp["batch_size"],
        "gradient_accumulation_steps": exp["gradient_accumulation_steps"],
        "effective_batch_size": exp["batch_size"] * exp["gradient_accumulation_steps"],
        "epochs": exp["epochs"],
        "max_length": exp["max_length"],
        "macro_f1": eval_result.get("eval_macro_f1", 0),
        "weighted_f1": eval_result.get("eval_weighted_f1", 0),
        "accuracy": eval_result.get("eval_accuracy", 0),
        "macro_precision": eval_result.get("eval_macro_precision", 0),
        "macro_recall": eval_result.get("eval_macro_recall", 0),
        "training_time_seconds": training_time,
        "best_checkpoint": best_checkpoint
    }
    
    results.append(experiment_result)
    
    print("\nExperiment Result:")
    print(json.dumps(experiment_result, indent=2, ensure_ascii=False))

    current_score = experiment_result["macro_f1"]
    if current_score > best_score:
        best_score = current_score
        best_result = experiment_result.copy()
        best_experiment = exp.copy()
        best_model_path = best_checkpoint
        print("\n🏆 NEW BEST MODEL FOUND!")
        print(f"Macro F1: {best_score:.6f}")

    del trainer, model, tokenizer, ds_train, ds_val
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# 9. Compile Results & Save Best Configuration JSON

In [ ]:
print("\n" + "=" * 70)
print("BENCHMARK COMPLETE")
print("=" * 70)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["macro_f1", "macro_recall", "weighted_f1"], ascending=False).reset_index(drop=True)

display(results_df)

results_csv = os.path.join(FINAL_DIR, "benchmark_results_5_experiments.csv")
results_json = os.path.join(FINAL_DIR, "benchmark_results_5_experiments.json")
results_df.to_csv(results_csv, index=False)
with open(results_json, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

best_config = {
    "experiment_id": best_experiment["experiment_id"],
    "model_name": best_experiment["model_name"],
    "learning_rate": best_experiment["learning_rate"],
    "batch_size": best_experiment["batch_size"],
    "gradient_accumulation_steps": best_experiment["gradient_accumulation_steps"],
    "epochs": best_experiment["epochs"],
    "max_length": best_experiment["max_length"],
    "macro_f1": float(best_result["macro_f1"]),
    "weighted_f1": float(best_result["weighted_f1"]),
    "accuracy": float(best_result["accuracy"]),
    "best_checkpoint": best_model_path,
    "num_labels": NUM_LABELS,
    "label2id": LABEL2ID,
    "id2label": {str(k): v for k, v in ID2LABEL.items()}
}

best_config_path = os.path.join(FINAL_DIR, "best_experiment_config.json")
with open(best_config_path, "w", encoding="utf-8") as f:
    json.dump(best_config, f, indent=2, ensure_ascii=False)


# 10. Final Export for Pipeline Contract\nLoads the best checkpoint weights and formally exports them to `outputs/training/best_model/` so Notebook 06 can automatically consume them.

In [ ]:
print("\n" + "=" * 70)
print("EXPORTING BEST MODEL FOR INFERENCE PIPELINE")
print("=" * 70)

if best_model_path and os.path.exists(best_model_path):
    print(f"Loading best checkpoint from: {best_model_path}")
    
    # Reload the winning model & tokenizer
    final_model = AutoModelForSequenceClassification.from_pretrained(best_model_path)
    final_tokenizer = AutoTokenizer.from_pretrained(best_model_path)
    
    # Save them to the unified pipeline handoff directory
    print(f"Saving to pipeline directory: {BEST_MODEL_EXPORT_DIR}")
    final_model.save_pretrained(BEST_MODEL_EXPORT_DIR)
    final_tokenizer.save_pretrained(BEST_MODEL_EXPORT_DIR)
    
    # Save the label mapping as required by Notebook 06
    mapping_path = os.path.join(BEST_MODEL_EXPORT_DIR, "label_mapping.json")
    with open(mapping_path, "w", encoding="utf-8") as f:
        json.dump(LABEL2ID, f, indent=4)
        
    # Save the metadata for reproducibility
    metadata_path = os.path.join(BEST_MODEL_EXPORT_DIR, "model_metadata.json")
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(best_config, f, indent=4)
        
    print("✅ Model, Tokenizer, and Mappings successfully exported!")
else:
    print("❌ ERROR: Best model path not found. Export failed.")

if torch.cuda.is_available():
    print("\nFinal GPU memory usage:")
    print(f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
